In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [2]:
dfmdock_result = pd.read_csv('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/dfmdock_perturb_tr_likelihood/dfmdock_inference/results/csv_files/db5_test_DFMDock_model_0_0.5_120_samples_40_steps_dips_.csv')
flowtime_result = pd.read_csv('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/dfmdock_perturb_tr_likelihood/flowtime_dfmdock_perturb_tr.csv')
difftime_result = pd.read_csv('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/dfmdock_perturb_tr_likelihood/difftime_dfmdock_perturb_tr.csv')
diffspace_result = pd.read_csv('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/dfmdock_perturb_tr_likelihood/diffspace_dfmdock_perturb_tr.csv')
rosetta_score = pd.read_csv('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/dfmdock_perturb_tr_likelihood/rosetta_score/refine_score.sc', delim_whitespace=True, skiprows=1)

/tmp/ipykernel_223891/1425351200.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  rosetta_score = pd.read_csv('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/dfmdock_perturb_tr_likelihood/rosetta_score/refine_score.sc', delim_whitespace=True, skiprows=1)


In [4]:
unique_ids = flowtime_result['id'].apply(lambda x: x.split('_')[0]).unique()
unique_ids.sort()
results = {}
for _id in tqdm(unique_ids):
    dockq = []
    i_rmsd = []
    flowtime_nll = []
    difftime_nll = []
    diffspace_nll = []
    rosetta_Isc = []
    dfmdock_data = dfmdock_result[dfmdock_result['id'] == _id]
    for idx in range(120):
        rosetta_score_entry = rosetta_score[rosetta_score['description'] == f'{_id}_p{idx}_0001']
        if rosetta_score_entry.empty:
            continue
        else:
            rosetta_Isc.append(rosetta_score_entry['I_sc'].values[0])
            entry = dfmdock_data[dfmdock_data['index'] == idx]
            dockq.append(entry['DockQ'].values[0])
            i_rmsd.append(entry['i_rmsd'].values[0])
            flowtime_entry = flowtime_result[flowtime_result['id'] == f'{_id}_p{idx}.pdb']
            flowtime_nll.append(flowtime_entry['nll'].values[0])
            difftime_ids = difftime_result['id'].apply(lambda x: Path(x).stem)
            difftime_entry = difftime_result[difftime_ids == f'{_id}_p{idx}']
            difftime_nll.append(difftime_entry['nll'].values[0])
            diffspace_ids = diffspace_result['id'].apply(lambda x: Path(x).stem)
            diffspace_entry = diffspace_result[diffspace_ids == f'{_id}_p{idx}']
            diffspace_nll.append(diffspace_entry['nll'].values[0])
    results[_id] = {
        'dockq': np.array(dockq),
        'i_rmsd': np.array(i_rmsd),
        'flowtime_nll': np.array(flowtime_nll),
        'difftime_nll': np.array(difftime_nll),
        'diffspace_nll': np.array(diffspace_nll),
        'rosetta_Isc': np.array(rosetta_Isc)
    }

100%|██████████| 25/25 [01:46<00:00,  4.24s/it]


In [5]:
results = dict(sorted(results.items()))

In [6]:
len(results)

25

In [7]:
import pickle

In [8]:
with open('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/dfmdock_perturb_tr_likelihood/dfmdock_perturb_tr_likelihood_all.pkl', 'wb') as f:
    pickle.dump(results, f)

In [10]:
_id = '2SIC'
flowtime_ids = flowtime_result['id'].apply(lambda x: x.split('_')[0])
flowtime_entry = flowtime_result[flowtime_ids == _id]
flowtime_entry.sort_values('nll', inplace=True)
flowtime_entry.head(1)

/tmp/ipykernel_223891/3116434734.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flowtime_entry.sort_values('nll', inplace=True)


,id,nll,prior_logp,delta_logp
1831,2SIC_p31.pdb,4.355016,-13.072932,0.007886


In [13]:
dfmdock_result.query('id == "2SIC" and index == 31')

,id,index,c_rmsd,i_rmsd,l_rmsd,fnat,DockQ,energy,num_clashes
1831,2SIC,31,0.815644,1.030434,2.511486,0.628571,0.742556,-49.862911,0


In [12]:
_id = '2SIC'
rosetta_ids = rosetta_score['description'].apply(lambda x: x.split('_')[0])
rosetta_entry = rosetta_score[rosetta_ids == _id]
rosetta_entry = rosetta_entry[rosetta_entry['I_sc'] > -10000]
rosetta_entry = rosetta_entry.sort_values('I_sc')
rosetta_entry.head(1)

,SCORE:,total_score,rms,CAPRI_rank,Fnat,I_sc,Irms,Irms_leg,dslf_fa13,fa_atr,...,hbond_sc,hbond_sr_bb,lk_ball_wtd,omega,p_aa_pp,pro_close,rama_prepro,ref,yhh_planarity,description
1797,SCORE:,15.08,2.592,3.0,0.836,-49.383,0.989,0.575,0.0,-2140.574,...,-26.65,-87.365,-42.907,46.889,-54.012,46.075,139.14,198.169,0.0,2SIC_p5_0001


In [14]:
dfmdock_result.query('id == "2SIC" and index == 5')

,id,index,c_rmsd,i_rmsd,l_rmsd,fnat,DockQ,energy,num_clashes
1805,2SIC,5,0.453895,0.54053,1.312957,0.685714,0.84916,-49.577812,0


In [15]:
dfmdock_result.query('id == "2SIC"').sort_values('DockQ', ascending=False).head(3)

,id,index,c_rmsd,i_rmsd,l_rmsd,fnat,DockQ,energy,num_clashes
1890,2SIC,90,0.218292,0.231765,0.528522,0.971429,0.981420,-59.258389,0
1903,2SIC,103,0.234558,0.296954,0.734824,0.971429,0.975432,-55.982384,0
1830,2SIC,30,0.262160,0.321357,0.786858,0.971429,0.973016,-56.583332,0


In [16]:
_id = '1IRA'
flowtime_ids = flowtime_result['id'].apply(lambda x: x.split('_')[0])
flowtime_entry = flowtime_result[flowtime_ids == _id]
flowtime_entry.sort_values('nll', inplace=True)
flowtime_entry.head(1)

/tmp/ipykernel_223891/3447791019.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flowtime_entry.sort_values('nll', inplace=True)


,id,nll,prior_logp,delta_logp
455,1IRA_p95.pdb,4.414812,-13.236,-0.008436


In [17]:
dfmdock_result.query('id == "1IRA" and index == 95')

,id,index,c_rmsd,i_rmsd,l_rmsd,fnat,DockQ,energy,num_clashes
455,1IRA,95,0.375864,0.402507,0.816255,0.904762,0.942819,-49.91972,0


In [18]:
_id = '1IRA'
rosetta_ids = rosetta_score['description'].apply(lambda x: x.split('_')[0])
rosetta_entry = rosetta_score[rosetta_ids == _id]
rosetta_entry = rosetta_entry[rosetta_entry['I_sc'] > -10000]
rosetta_entry = rosetta_entry.sort_values('I_sc')
rosetta_entry.head(1)

,SCORE:,total_score,rms,CAPRI_rank,Fnat,I_sc,Irms,Irms_leg,dslf_fa13,fa_atr,...,hbond_sc,hbond_sr_bb,lk_ball_wtd,omega,p_aa_pp,pro_close,rama_prepro,ref,yhh_planarity,description
401,SCORE:,-15.119,1.265,3.0,0.845,-59.825,0.706,0.58,-0.303,-2455.717,...,-26.361,-48.17,-51.921,-3.015,-57.021,30.936,98.772,111.911,0.0,1IRA_p22_0001


In [19]:
dfmdock_result.query('id == "1IRA" and index == 22')

,id,index,c_rmsd,i_rmsd,l_rmsd,fnat,DockQ,energy,num_clashes
382,1IRA,22,0.584378,0.62372,1.296573,0.761905,0.863918,-47.861591,1


In [20]:
dfmdock_result.query('id == "1IRA"').sort_values('DockQ', ascending=False).head(3)

,id,index,c_rmsd,i_rmsd,l_rmsd,fnat,DockQ,energy,num_clashes
455,1IRA,95,0.375864,0.402507,0.816255,0.904762,0.942819,-49.919720,0
382,1IRA,22,0.584378,0.623720,1.296573,0.761905,0.863918,-47.861591,1
396,1IRA,36,1.133533,1.215286,2.563667,0.619048,0.713127,-41.494564,0


In [23]:
gt_flowtime = pd.read_csv('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/dfmdock_perturb_tr_likelihood/ground_truth_energy/flowtime/flowtime_dfmdock_tr_gt.csv')
gt_rosetta_score = pd.read_csv('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/old/dfmdock_tr_likelihood/groud_truth_energy/rosetta/refine_score.sc', delim_whitespace=True, skiprows=1)

/tmp/ipykernel_223891/217488937.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  gt_rosetta_score = pd.read_csv('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/old/dfmdock_tr_likelihood/groud_truth_energy/rosetta/refine_score.sc', delim_whitespace=True, skiprows=1)


In [25]:
gt_flowtime_dict = {row['id'].split('.')[0]: row['nll'] for _, row in gt_flowtime.iterrows()}

In [26]:
gt_rosetta_score

,SCORE:,total_score,rms,CAPRI_rank,Fnat,I_sc,Irms,Irms_leg,dslf_fa13,fa_atr,...,hbond_sc,hbond_sr_bb,lk_ball_wtd,omega,p_aa_pp,pro_close,rama_prepro,ref,yhh_planarity,description
0,SCORE:,214.504,3.295,3.0,0.796,-18.350,0.788,0.528,13.199,-2173.009,...,-17.286,-33.940,-47.382,1.112,-56.992,45.914,135.574,168.173,0.0,1AVX_0005
1,SCORE:,214.626,3.569,3.0,0.796,-20.386,0.887,0.562,13.199,-2168.972,...,-20.247,-33.940,-47.166,1.112,-56.992,45.914,135.574,168.173,0.0,1AVX_0004
2,SCORE:,212.621,4.383,3.0,0.776,-24.102,0.927,0.674,13.199,-2172.288,...,-20.812,-33.940,-46.167,1.112,-56.992,45.914,135.574,168.173,0.0,1AVX_0003
3,SCORE:,214.956,3.580,3.0,0.776,-22.367,0.849,0.543,13.199,-2177.461,...,-19.310,-33.940,-47.501,1.112,-56.992,45.914,135.574,168.173,0.0,1AVX_0002
4,SCORE:,90.045,0.445,3.0,0.951,-38.478,0.448,0.189,33.483,-1711.767,...,-26.468,-22.007,-32.331,32.106,-64.126,19.817,123.629,164.854,0.0,1HCF_0004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,SCORE:,4173.363,3.640,2.0,0.508,-41.400,1.409,1.143,0.000,-16854.961,...,-203.467,-1034.882,-310.725,29.701,-248.640,219.881,932.472,804.126,0.0,1N2C_0003
93,SCORE:,4289.012,1.804,3.0,0.639,-55.671,0.887,0.703,0.000,-16903.197,...,-209.091,-1034.882,-302.173,29.701,-248.640,220.252,932.472,804.126,0.0,1N2C_0004
94,SCORE:,4053.820,1.832,3.0,0.656,-47.941,0.824,0.659,0.000,-16872.135,...,-208.393,-1034.882,-304.741,29.701,-248.640,227.788,932.472,804.126,0.0,1N2C_0002
95,SCORE:,3936.537,2.048,3.0,0.730,-57.320,0.758,0.602,0.000,-16911.854,...,-218.466,-1034.882,-307.626,29.701,-248.640,216.651,932.472,804.126,0.0,1N2C_0005


In [27]:
gt_rosetta_score_dict = {}
for _, row in gt_rosetta_score.iterrows():
    key = row['description'].split('_')[0]
    if key not in gt_rosetta_score_dict:
        gt_rosetta_score_dict[key] = []
    gt_rosetta_score_dict[key].append(row['I_sc'])
gt_rosetta_score_dict = {k: min(v) for k, v in gt_rosetta_score_dict.items()}

In [28]:
import pickle

In [29]:
with open('/home/dxu39/scr4_jgray21/dxu39/projects/diffenergy/dfmdock_perturb_tr_likelihood/dfmdock_perturb_tr_likelihood_gt.pkl', 'wb') as f:
    pickle.dump({'flowtime': gt_flowtime_dict, 'rosetta': gt_rosetta_score_dict}, f)